# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [9]:
# Write your code below.
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Get PRICE_DATA directory
price_data_dir = os.getenv("PRICE_DATA")
if not price_data_dir:
    raise ValueError("PRICE_DATA directory is not set in the environment variable.")
print(f"PRICE_DATA directory: {price_data_dir}")



PRICE_DATA directory: ../../05_src/data/prices/


In [10]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [11]:
import os
from glob import glob

# Write your code below.
# Find all Parquet files in the directory
parquet_files = glob(os.path.join(price_data_dir, "**/*.parquet"), recursive = True)
dd_px = dd.read_parquet(parquet_files).set_index("Ticker")



For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [12]:
print(dd_px.columns)

Index(['Date', 'Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'Year'], dtype='object', name='Price')


In [13]:
# Write your code below.
# Add lags for variables Close and Adj_Close
dd_feat = dd_px.groupby("Ticker", group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x["Close"].shift(1)))

# Add returns based on close
dd_feat = dd_feat.assign(returns = dd_feat["Close"] / dd_feat["Close_lag_1"] - 1)

# Addmthe following range 'hi_lo_range'
dd_feat = dd_feat.assign(hi_lo_range =dd_feat["High"] - dd_feat["Low"])

dd_feat.compute()

<ipython-input-13-90c1f8c4bd60>:3: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_feat = dd_px.groupby("Ticker", group_keys=False).apply(


Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,returns,hi_lo_range
Ticker,,,,,,,,,,,
DOV,2017-01-03 00:00:00+00:00,53.950462,61.728596,62.625202,60.920841,61.639744,1637998.0,2017,NaN,NaN,1.704361
DOV,2017-01-04 00:00:00+00:00,54.204605,62.019386,62.075928,61.445881,61.882069,1165948.0,2017,61.728596,0.004711,0.630047
DOV,2017-01-05 00:00:00+00:00,54.028118,61.817448,62.431339,61.074314,61.793217,1155178.0,2017,62.019386,-0.003256,1.357025
DOV,2017-01-06 00:00:00+00:00,54.868225,62.778675,63.780293,62.390953,62.463650,3123350.0,2017,61.817448,0.015549,1.389339
DOV,2017-01-09 00:00:00+00:00,54.162258,61.970921,62.924072,61.744751,62.705978,1271055.0,2017,62.778675,-0.012867,1.179321
...,...,...,...,...,...,...,...,...,...,...,...
CTLT,2002-12-24 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2002,NaN,NaN,NaN
CTLT,2002-12-26 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2002,NaN,NaN,NaN
CTLT,2002-12-27 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,2002,NaN,NaN,NaN


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [14]:
# Write your code below.
# Write your code below.
import pandas as pd

# Convert Dask DataFrame to Pandas DataFrame
pdf = dd_feat.compute()

pdf = pdf.sort_values(["Ticker", "Date"])

pdf = pdf.groupby("Ticker", group_keys = False).apply(
    lambda x: x.assign(returns_ma_10 = x["returns"].rolling(10, min_periods = 1).mean()))

pdf.head()


Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,returns,hi_lo_range,returns_ma_10
Ticker,,,,,,,,,,,,
A,2000-01-03 00:00:00+00:00,43.382843,51.502148,56.464592,48.193848,56.330471,4674353.0,2000,40.908440,0.258961,8.270744,0.258961
A,2000-01-04 00:00:00+00:00,40.068890,47.567955,49.266811,46.316166,48.730328,4765083.0,2000,51.502148,-0.076389,2.950645,0.091286
A,2000-01-05 00:00:00+00:00,37.583389,44.617310,47.567955,43.141991,47.389126,5758642.0,2000,47.567955,-0.062030,4.425964,0.040181
A,2000-01-06 00:00:00+00:00,36.152363,42.918453,44.349072,41.577251,44.080830,2534434.0,2000,44.617310,-0.038076,2.771820,0.020617
A,2000-01-07 00:00:00+00:00,39.165070,46.494991,47.165592,42.203148,42.247852,2819626.0,2000,42.918453,0.083333,4.962444,0.033160


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.